In [ ]:
# Cell 1 — Imports and path setup
"""
01_prepare_data.ipynb
=====================
Download all data sources, merge them, create canonical splits, and build DataLoaders.

CLI equivalent: `python scripts/prepare_data.py --output-dir notebooks/data/processed --split-mode solute_scaffold`
Canonical outputs: scaffold, solute, and solvent split families plus `split_manifest.json`
in `notebooks/data/processed/`.
Reference docs: `docs/data_preparation.md` and `docs/script_reference.md`.
"""

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
NOTEBOOK_FIG_DIR = FIGURES_DIR / "notebooks"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / "notebooks"
NOTEBOOK_FIG_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from tgnn_solv.data import (
    load_bigsoldb,
    load_melting_points,
    load_fusion_enthalpies,
    load_hansen,
    load_idac,
    DataBuilder,
    filter_for_sle,
    scaffold_split,
    make_loaders,
    PROCESSED_DIR,
)
import json

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data dir: {PROCESSED_DIR}")
print("All imports OK")


In [ ]:
# Cell 2 — Load primary source
bigsoldb = load_bigsoldb()
print(f"\nShape: {bigsoldb.shape}")
bigsoldb.head()

In [ ]:
# Cell 3 — Filter for SLE compatibility
bigsoldb = filter_for_sle(bigsoldb, x2_max=0.98)
print(f"After SLE filter: {len(bigsoldb):,}")

In [ ]:
# Cell 4 — Load auxiliary sources
mp_data = load_melting_points()
dh_data = load_fusion_enthalpies()
hansen_data = load_hansen()
idac_data = load_idac()

In [ ]:
# Cell 5 — Build unified dataset
builder = DataBuilder()
builder.add_mp(mp_data)
builder.add_dh(dh_data)
builder.add_hansen(hansen_data)
builder.add_gamma(idac_data)

unified = builder.build(bigsoldb)
print(f"\nUnified shape: {unified.shape}")
unified.head()

In [ ]:
# Cell — Data quality check
import matplotlib.pyplot as plt

sol_data = unified[unified["has_solubility"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(sol_data["ln_x2"], bins=100, edgecolor="white")
axes[0].set_xlabel("ln(x₂)")
axes[0].set_title(f"Distribution (n={len(sol_data):,})")

sol_counts = sol_data["solute_smiles"].value_counts()
axes[1].hist(sol_counts.values, bins=50, edgecolor="white")
axes[1].set_xlabel("Records per solute")
axes[1].set_title(f"Solute frequency (n={len(sol_counts):,})")

plt.tight_layout()
plt.show()

In [ ]:
print(f"\nSolutes with only 1 record: "
      f"{(sol_counts == 1).sum()} / {len(sol_counts)}")
print(f"Solutes with >50 records: "
      f"{(sol_counts > 50).sum()}")
non_298 = (sol_data["temperature"] - 298.15).abs() > 1
print(f"\nRecords at T≠298K: {non_298.sum()} ({non_298.sum()/len(sol_data)*100:.1f}%)")

In [ ]:
# Cell 6 — Splits (scaffold + solute + solvent)
# `scripts/prepare_data.py` now writes all canonical split families in one run.
split_outputs = {}
for split_mode in ("solute_scaffold", "solute", "solvent"):
    split_train, split_val, split_test = scaffold_split(unified, mode=split_mode)
    suffix = {
        "solute_scaffold": "",
        "solute": "_solute",
        "solvent": "_solvent",
    }[split_mode]

    (PROCESSED_DIR / f"train{suffix}.csv").parent.mkdir(parents=True, exist_ok=True)
    split_train.to_csv(PROCESSED_DIR / f"train{suffix}.csv", index=False)
    split_val.to_csv(PROCESSED_DIR / f"val{suffix}.csv", index=False)
    split_test.to_csv(PROCESSED_DIR / f"test{suffix}.csv", index=False)

    split_outputs[split_mode] = {
        "train": str(PROCESSED_DIR / f"train{suffix}.csv"),
        "val": str(PROCESSED_DIR / f"val{suffix}.csv"),
        "test": str(PROCESSED_DIR / f"test{suffix}.csv"),
    }

    if split_mode == "solute_scaffold":
        train_df, val_df, test_df = split_train, split_val, split_test
    elif split_mode == "solute":
        train_sol_df, val_sol_df, test_sol_df = split_train, split_val, split_test
    else:
        train_slv_df, val_slv_df, test_slv_df = split_train, split_val, split_test

manifest = {
    "primary_split_mode": "solute_scaffold",
    "splits": split_outputs,
}
(PROCESSED_DIR / "split_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Saved scaffold, solute, and solvent splits to {PROCESSED_DIR}")


In [ ]:
# Cell 7 — Create DataLoaders (smoke test)
# Uses scaffold split by default (train.csv/val.csv/test.csv).
# For fair baseline comparison, swap in the `*_solute.csv` split.
train_loader, val_loader, test_loader = make_loaders(
    train_df, val_df, test_df, batch_size=64,
)

# Verify one batch
for sol_b, slv_b, tgt in train_loader:
    B = tgt["T"].shape[0]
    print(f"Batch B={B}")
    print(f"  Solute atoms:    {sol_b.x.shape[0]}")
    print(f"  Solvent atoms:   {slv_b.x.shape[0]}")
    print(f"  has_solubility:  {tgt['has_solubility'].sum()}/{B}")
    print(f"  T_m_mask:        {tgt['T_m_mask'].sum()}/{B}")
    print(f"  dH_mask:         {tgt['dH_mask'].sum()}/{B}")
    print(f"  hansen_mask:     {tgt['hansen_mask'].sum()}/{B}")
    print(f"  gamma_mask:      {tgt['gamma_mask'].sum()}/{B}")
    break

print("Data pipeline complete!")
